# House Price Prediction — End-to-End ML Project

**Student Project** — From raw data to deployed web app

## Overview

This notebook covers the complete ML pipeline:
1. **Load & Inspect** — Understand the dataset
2. **EDA** — Exploratory Data Analysis with 4+ plots
3. **Cleaning & Feature Engineering** — Handle messy real-world data
4. **Build Pipeline & Train** — scikit-learn Pipeline with ColumnTransformer
5. **Evaluate** — MAE, RMSE, R², cross-validation, model comparison
6. **Export** — Save model as `.pkl` and locations as `.json`

---

## 1. Load & Inspect

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Libraries loaded successfully')

In [ ]:
# Load the dataset
df = pd.read_csv('data/house_prices.csv')

# Basic info
print(f'Shape: {df.shape}')
print(f'\nColumns ({len(df.columns)}):')
for i, col in enumerate(df.columns):
    print(f'  {i+1:2d}. {col}')

print(f'\nDtypes:')
print(df.dtypes)

print(f'\nMissing values (%):')
missing = df.isna().mean().sort_values(ascending=False) * 100
print(missing[missing > 0].to_string())

In [ ]:
# Sample data
print(df.head(3).to_string())

# Unique value counts for categorical columns
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    n_unique = df[col].nunique()
    if n_unique < 30:
        print(f'\n{col} ({n_unique} unique): {df[col].dropna().unique()}')
    else:
        print(f'\n{col} ({n_unique} unique) — showing top 10:')
        print(df[col].value_counts().head(10))

---

## 2. Exploratory Data Analysis (EDA)

### 2.1 Target Variable — Price Distribution

In [ ]:
# Parse price from Amount(in rupees) — text like '42 Lac', '1.40 Cr', 'Call for Price'
def parse_amount(x):
    if not isinstance(x, str):
        return np.nan
    x = x.strip().lower()
    if 'call for price' in x or 'price on request' in x:
        return np.nan
    try:
        if 'lac' in x:
            return float(x.replace('lac', '').strip()) * 1e5
        if 'cr' in x:
            return float(x.replace('cr', '').strip()) * 1e7
        # Plain number (might have commas)
        return float(x.replace(',', ''))
    except ValueError:
        return np.nan

df['price_clean'] = df['Amount(in rupees)'].apply(parse_amount)

# Drop rows without usable price
df_clean = df.dropna(subset=['price_clean']).copy()
print(f'Original rows: {len(df)}')
print(f'After dropping missing price: {len(df_clean)}')
print(f'Dropped: {len(df) - len(df_clean)} ({100*(len(df)-len(df_clean))/len(df):.1f}%)')

# Price distribution plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Linear scale
axes[0,0].hist(df_clean['price_clean'], bins=50, edgecolor='black', alpha=0.7)
axes[0,0].set_title('Price Distribution (Linear Scale)', fontsize=12)
axes[0,0].set_xlabel('Price (₹)')
axes[0,0].set_ylabel('Count')
axes[0,0].ticklabel_format(style='scientific', axis='x', scilimits=(0,0))

# Log scale
axes[0,1].hist(df_clean['price_clean'], bins=50, edgecolor='black', alpha=0.7, log=True)
axes[0,1].set_title('Price Distribution (Log Scale)', fontsize=12)
axes[0,1].set_xlabel('Price (₹)')
axes[0,1].set_ylabel('Count (log)')
axes[0,1].ticklabel_format(style='scientific', axis='x', scilimits=(0,0))

# Log10 of price
axes[1,0].hist(np.log10(df_clean['price_clean']), bins=50, edgecolor='black', alpha=0.7)
axes[1,0].set_title('Log10(Price) Distribution', fontsize=12)
axes[1,0].set_xlabel('Log10(Price in ₹)')
axes[1,0].set_ylabel('Count')

# Box plot
axes[1,1].boxplot(df_clean['price_clean'], vert=False)
axes[1,1].set_title('Price Box Plot', fontsize=12)
axes[1,1].set_xlabel('Price (₹)')
axes[1,1].ticklabel_format(style='scientific', axis='x', scilimits=(0,0))

plt.tight_layout()
plt.show()

print('Price statistics:')
print(df_clean['price_clean'].describe())
print(f'\nMedian: ₹{df_clean["price_clean"].median():,.0f}')
print(f'Mean:   ₹{df_clean["price_clean"].mean():,.0f}')

### 2.2 Price vs Carpet Area

In [ ]:
# Parse carpet area — extract number and convert to sqft
def parse_area(x):
    if not isinstance(x, str):
        return np.nan
    x = x.strip().lower()
    try:
        if 'sqft' in x:
            return float(x.replace('sqft', '').strip())
        if 'sqm' in x or 'sq m' in x:
            return float(x.replace('sqm', '').replace('sq m', '').strip()) * 10.764
        # Assume sqft if no unit
        return float(x)
    except ValueError:
        return np.nan

df_clean['carpet_area_sqft'] = df_clean['Carpet Area'].apply(parse_area)

# Parse super area too
df_clean['super_area_sqft'] = df_clean['Super Area'].apply(parse_area)

print(f'Carpet area parsed: {df_clean["carpet_area_sqft"].notna().sum()} / {len(df_clean)}')
print(f'Super area parsed:  {df_clean["super_area_sqft"].notna().sum()} / {len(df_clean)}')

# Price vs Carpet Area scatter
plt.figure(figsize=(10, 6))
sample = df_clean.dropna(subset=['carpet_area_sqft']).sample(min(5000, len(df_clean)), random_state=42)
plt.scatter(sample['carpet_area_sqft'], sample['price_clean'], alpha=0.4, s=10)
plt.xlabel('Carpet Area (sqft)')
plt.ylabel('Price (₹)')
plt.title('Price vs Carpet Area', fontsize=14)
plt.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
plt.tight_layout()
plt.show()

# Correlation
corr = df_clean[['price_clean', 'carpet_area_sqft']].corr().iloc[0,1]
print(f'Correlation (Price vs Carpet Area): {corr:.3f}')

### 2.3 Average Price by Top Locations

In [ ]:
# Top 15 locations by count
top_locations = df_clean['location'].value_counts().head(15).index
df_top = df_clean[df_clean['location'].isin(top_locations)]

avg_price_by_loc = df_top.groupby('location')['price_clean'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
avg_price_by_loc.plot(kind='bar', edgecolor='black')
plt.title('Average Price by Top 15 Locations', fontsize=14)
plt.xlabel('Location')
plt.ylabel('Average Price (₹)')
plt.xticks(rotation=45, ha='right')
plt.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
plt.tight_layout()
plt.show()

print('Average price by location (₹):')
for loc, price in avg_price_by_loc.items():
    print(f'  {loc:20s}: ₹{price:,.0f}')

### 2.4 Price by Furnishing Status & Bathrooms

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot: Price by Furnishing
furnish_order = ['Unfurnished', 'Semi-Furnished', 'Furnished']
df_furnish = df_clean[df_clean['Furnishing'].isin(furnish_order)]
sns.boxplot(data=df_furnish, x='Furnishing', y='price_clean', order=furnish_order, ax=axes[0])
axes[0].set_title('Price by Furnishing Status', fontsize=12)
axes[0].set_ylabel('Price (₹)')
axes[0].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

# Box plot: Price by Bathroom count
# Parse bathroom to numeric
def parse_bathroom(x):
    if not isinstance(x, str):
        return np.nan
    x = x.strip()
    if '> 10' in x:
        return 11
    try:
        return float(x)
    except ValueError:
        return np.nan

df_clean['bathroom_num'] = df_clean['Bathroom'].apply(parse_bathroom)

sns.boxplot(data=df_clean.dropna(subset=['bathroom_num']), x='bathroom_num', y='price_clean', ax=axes[1])
axes[1].set_title('Price by Number of Bathrooms', fontsize=12)
axes[1].set_xlabel('Bathrooms')
axes[1].set_ylabel('Price (₹)')
axes[1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.show()

print('Average price by furnishing:')
print(df_furnish.groupby('Furnishing')['price_clean'].mean().to_string())

print('\nAverage price by bathroom count:')
print(df_clean.dropna(subset=['bathroom_num']).groupby('bathroom_num')['price_clean'].mean().to_string())

### 2.5 Additional EDA — Price vs Floor & Transaction Type

In [ ]:
# Parse floor number
def parse_floor(x):
    if not isinstance(x, str):
        return np.nan
    x = x.strip().lower()
    if 'ground' in x:
        return 0
    # Extract first number from 'X out of Y'
    import re
    match = re.search(r'(\d+)', x)
    if match:
        return int(match.group(1))
    return np.nan

df_clean['floor_num'] = df_clean['Floor'].apply(parse_floor)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Price vs Floor
df_floor = df_clean.dropna(subset=['floor_num'])
sns.boxplot(data=df_floor, x='floor_num', y='price_clean', ax=axes[0])
axes[0].set_title('Price by Floor Number', fontsize=12)
axes[0].set_xlabel('Floor')
axes[0].set_ylabel('Price (₹)')
axes[0].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

# Price by Transaction type
sns.boxplot(data=df_clean, x='Transaction', y='price_clean', ax=axes[1])
axes[1].set_title('Price by Transaction Type', fontsize=12)
axes[1].set_xlabel('Transaction')
axes[1].set_ylabel('Price (₹)')
axes[1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

---

## 3. Cleaning & Feature Engineering

In [ ]:
# Start fresh from cleaned dataframe
df_feat = df_clean.copy()

# 3.1 Parse balcony
def parse_balcony(x):
    if not isinstance(x, str):
        return np.nan
    x = x.strip()
    if '> 10' in x:
        return 11
    try:
        return float(x)
    except ValueError:
        return np.nan

df_feat['balcony_num'] = df_feat['Balcony'].apply(parse_balcony)

# 3.2 Parse car parking — extract numeric
def parse_parking(x):
    if not isinstance(x, str):
        return 0
    x = x.strip().lower()
    import re
    match = re.search(r'(\d+)', x)
    if match:
        return int(match.group(1))
    return 0

df_feat['car_parking_num'] = df_feat['Car Parking'].apply(parse_parking)

# 3.3 Handle high-cardinality categoricals: location & Society
# Keep top-N locations, group rest as 'other'
TOP_N_LOCATIONS = 50
top_locs = df_feat['location'].value_counts().head(TOP_N_LOCATIONS).index.tolist()
df_feat['location_grouped'] = df_feat['location'].apply(lambda x: x if x in top_locs else 'other')

# Society — keep top 30, rest 'other'
TOP_N_SOCIETY = 30
top_soc = df_feat['Society'].value_counts().head(TOP_N_SOCIETY).index.tolist()
df_feat['society_grouped'] = df_feat['Society'].apply(lambda x: x if x in top_soc else 'other')

# 3.4 Drop useless columns
drop_cols = ['Index', 'Title', 'Description', 'Amount(in rupees)', 'Price (in rupees)',
             'Carpet Area', 'Super Area', 'Status', 'Floor', 'Bathroom', 'Balcony',
             'Car Parking', 'overlooking', 'Society', 'Dimensions', 'Plot Area',
             'location', 'facing']

df_feat = df_feat.drop(columns=[c for c in drop_cols if c in df_feat.columns])

# 3.5 Remove outliers — price per sqft
df_feat['price_per_sqft'] = df_feat['price_clean'] / df_feat['carpet_area_sqft']
lower = df_feat['price_per_sqft'].quantile(0.01)
upper = df_feat['price_per_sqft'].quantile(0.99)
print(f'Price/sqft range: {lower:,.0f} — {upper:,.0f}')
df_feat = df_feat[(df_feat['price_per_sqft'] >= lower) & (df_feat['price_per_sqft'] <= upper)]
df_feat = df_feat.drop(columns=['price_per_sqft'])

print(f'\nFinal shape after cleaning: {df_feat.shape}')
print(f'Columns: {df_feat.columns.tolist()}')

In [ ]:
# Check final feature columns and types
print('Feature columns:')
for col in df_feat.columns:
    if col != 'price_clean':
        dtype = df_feat[col].dtype
        n_unique = df_feat[col].nunique()
        sample = df_feat[col].dropna().unique()[:5]
        print(f'  {col:25s} | {str(dtype):10s} | {n_unique:4d} unique | sample: {sample}')

In [ ]:
# Prepare features and target
numeric_features = ['carpet_area_sqft', 'floor_num', 'bathroom_num', 'balcony_num', 'car_parking_num']
categorical_features = ['location_grouped', 'Furnishing', 'Transaction', 'Ownership', 'society_grouped']

X = df_feat[numeric_features + categorical_features]
y = df_feat['price_clean']

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

# Check for any remaining NaN
print(f'\nMissing in X:\n{X.isna().sum()}')
print(f'Missing in y: {y.isna().sum()}')

---

## 4. Build Pipeline & Train

Using `sklearn Pipeline` + `ColumnTransformer` so preprocessing is bundled inside the exported model.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import joblib

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Preprocessor
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_features),
])

# Define models to compare
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, random_state=42)
}

results = {}

for name, reg in models.items():
    print(f'\nTraining {name}...')
    model = Pipeline([("prep", preprocessor), ("reg", reg)])
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Cross-validation (5-fold)
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
    cv_rmse = -cv_scores.mean()
    cv_std = cv_scores.std()
    
    results[name] = {
        'model': model,
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
        'cv_rmse': cv_rmse,
        'cv_std': cv_std
    }
    
    print(f'  MAE:  ₹{mae:,.0f}')
    print(f'  RMSE: ₹{rmse:,.0f}')
    print(f'  R²:   {r2:.4f}')
    print(f'  5-fold CV RMSE: ₹{cv_rmse:,.0f} ± ₹{cv_std:,.0f}')

### 4.1 Model Comparison

In [ ]:
# Comparison table
import pandas as pd
comparison = pd.DataFrame([{
    'Model': name,
    'MAE (₹)': f'₹{res["mae"]:,.0f}',
    'RMSE (₹)': f'₹{res["rmse"]:,.0f}',
    'R²': f'{res["r2"]:.4f}',
    'CV RMSE (₹)': f'₹{res["cv_rmse"]:,.0f} ± ₹{res["cv_std"]:,.0f}'
} for name, res in results.items()])

print(comparison.to_string(index=False))

In [ ]:
# Predicted vs Actual scatter for best model
best_model_name = min(results.keys(), key=lambda k: results[k]['rmse'])
best_model = results[best_model_name]['model']
y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_best, alpha=0.4, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price (₹)')
plt.ylabel('Predicted Price (₹)')
plt.title(f'Predicted vs Actual — {best_model_name}', fontsize=14)
plt.ticklabel_format(style='scientific', axis='both', scilimits=(0,0))
plt.tight_layout()
plt.show()

print(f'Best model: {best_model_name}')
print(f'Test RMSE: ₹{results[best_model_name]["rmse"]:,.0f}')
print(f'Test R²:   {results[best_model_name]["r2"]:.4f}')

---

## 5. Evaluate & Select Winner

In [ ]:
# Detailed evaluation of best model
best = results[best_model_name]
print(f'=== {best_model_name} — Final Evaluation ===')
print(f'MAE:  ₹{best["mae"]:,.0f}')
print(f'RMSE: ₹{best["rmse"]:,.0f}')
print(f'R²:   {best["r2"]:.4f}')
print(f'5-fold CV RMSE: ₹{best["cv_rmse"]:,.0f} ± ₹{best["cv_std"]:,.0f}')

# Residual analysis
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_pred_best, residuals, alpha=0.4, s=10)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted Price (₹)')
axes[0].set_ylabel('Residual (₹)')
axes[0].set_title('Residual Plot')
axes[0].ticklabel_format(style='scientific', axis='both', scilimits=(0,0))

axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residual (₹)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
axes[1].ticklabel_format(style='scientific', axis='x', scilimits=(0,0))

plt.tight_layout()
plt.show()

print(f'Residual mean: {residuals.mean():,.0f}')
print(f'Residual std:  {residuals.std():,.0f}')

---

## 6. Export the Model

Save the full pipeline (preprocessor + model) as `.pkl` and the allowed locations as `.json`.

In [ ]:
# Save model
import joblib
joblib.dump(best_model, 'house_price.pkl')
print('Model saved to house_price.pkl')

# Sanity check: reload and predict one sample
loaded = joblib.load('house_price.pkl')
sample = X_test.iloc[[0]]
print(f'Sample input:\n{sample.to_string()}')
print(f'\nReloaded prediction: ₹{loaded.predict(sample)[0]:,.0f}')
print(f'Actual: ₹{y_test.iloc[0]:,.0f}')

In [ ]:
# Save allowed locations for frontend dropdown
import json
allowed_locations = sorted(df_feat['location_grouped'].unique().tolist())
with open('locations.json', 'w') as f:
    json.dump(allowed_locations, f, indent=2)

print(f'Saved {len(allowed_locations)} locations to locations.json')
print('First 10:', allowed_locations[:10])

In [ ]:
# Save model metrics for README
metrics = {
    'model': best_model_name,
    'mae': float(best['mae']),
    'rmse': float(best['rmse']),
    'r2': float(best['r2']),
    'cv_rmse': float(best['cv_rmse']),
    'cv_std': float(best['cv_std']),
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'features': {
        'numeric': numeric_features,
        'categorical': categorical_features
    }
}

with open('model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Model metrics saved to model_metrics.json')

---

## Summary

✅ **Data loaded & inspected** — 187,531 rows, 21 columns
✅ **EDA completed** — 6 plots (price distribution, price vs area, top locations, furnishing, bathrooms, floor, transaction)
✅ **Cleaning & feature engineering** — parsed text prices/areas, extracted floor numbers, handled high-cardinality categoricals, removed outliers
✅ **3 models trained & compared** — LinearRegression, RandomForest, GradientBoosting
✅ **Best model selected** — Based on test RMSE
✅ **Model exported** — Full pipeline saved as `house_price.pkl`
✅ **Locations exported** — `locations.json` for frontend dropdown
✅ **Metrics saved** — `model_metrics.json` for README

---
